<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a> 著作《<a href="http://mng.bz/orYv">从零构建大语言模型 (Build a Large Language Model From Scratch)</a>》配套补充代码<br>
<br>代码仓库：<a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# 第 3 章：编写注意力机制代码 (Coding Attention Mechanisms)


本 Notebook 中使用的软件包：


In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))

torch version: 2.7.1+cu118


- 本章将介绍作为 LLM 核心引擎的注意力机制（attention mechanisms）：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/01.webp?123" width="800px">

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/02.webp" width="800px">

&nbsp;
## 3.1 长序列建模面临的问题


- 本节无代码
- 由于源语言和目标语言在语法结构上存在差异，按词逐字翻译文本是不可行的：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/03.webp" width="800px">

- 在 Transformer 模型问世之前，编码器-解码器架构的循环神经网络（RNN）常用于机器翻译任务
- 在这种架构中，编码器处理来自源语言的 token 序列，利用隐藏状态（hidden state——神经网络内部的一种中间层）来生成整个输入序列的紧凑浓缩表示：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/04.webp" width="800px">

&nbsp;
## 3.2 利用注意力机制捕获数据依赖关系


- 本节无代码
- 通过注意力机制，网络中负责生成文本的解码器能够选择性地访问所有输入 token，这意味着在生成某个特定输出 token 时，某些输入 token 比其他 token 具有更高的重要性：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/05.webp" width="800px">

- Transformer 中的自注意力（Self-attention）是一种旨在增强输入表示的技术，它使序列中的每个位置都能与同一序列内的所有其他位置进行交互并衡量彼此的相关性


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/06.webp" width="300px">

&nbsp;
## 3.3 使用自注意力机制关注输入的不同部分


&nbsp;
### 3.3.1 [inputs]无可训练权重的简单自注意力机制


- 本节介绍一种非常简化的自注意力变体，其中不包含任何可训练权重
- 这纯粹是为了教学演示，并不是 Transformer 中实际使用的注意力机制
- 下一节（3.3.2 节）将对这种简单的注意力机制进行扩展，以实现真正的自注意力机制
- 假设给定输入序列 $x^{(1)}$ 到 $x^{(T)}$
  - 输入是一段文本（例如句子 "Your journey starts with one step"），并且已经按照第 2 章所述转换为 token 嵌入向量
  - 例如，$x^{(1)}$ 是代表单词 "Your" 的 d 维向量，以此类推
- **目标：** 为 $x^{(1)}$ 到 $x^{(T)}$ 中的每个输入序列元素 $x^{(i)}$ 计算上下文向量（context vector）$z^{(i)}$（其中 $z$ 和 $x$ 具有相同的维度）
    - 上下文向量 $z^{(i)}$ 是对所有输入 $x^{(1)}$ 到 $x^{(T)}$ 的加权求和
    - 该上下文向量是针对特定输入的“上下文特异性”表示
      - 我们不使用泛指任意输入 token 的占位符 $x^{(i)}$，而是具体以第二个输入 $x^{(2)}$ 为例
      - 延续这个具体示例，我们不考虑通用的占位符 $z^{(i)}$，而是考虑第二个输出上下文向量 $z^{(2)}$
      - 第二个上下文向量 $z^{(2)}$ 是针对第二个输入元素 $x^{(2)}$，对所有输入 $x^{(1)}$ 到 $x^{(T)}$ 进行加权求和的结果
      - 注意力权重（attention weights）决定了在计算 $z^{(2)}$ 时，每个输入元素对加权和的贡献程度
      - 简而言之，可以将 $z^{(2)}$ 视为 $x^{(2)}$ 的增强修改版，它融合了与当前任务相关的其他所有输入元素的信息


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/07.webp" width="800px">

- （请注意，为了保持画面整洁，图中的数值均截断保留至小数点后一位；类似地，后续其他图中的数值也可能做截断处理）


- 按照学术惯例，未归一化的注意力权重被称为**“注意力得分 (attention scores)”**，而总和归一化为 1 的注意力得分则被称为**“注意力权重 (attention weights)”**


- 下面的代码逐步演练上图所示的流程

<br>

- **步骤 1：** 计算未归一化的注意力得分 $\omega$
- 假设我们使用第二个输入 token 作为查询（query），即 $q^{(2)} = x^{(2)}$，我们通过点积计算未归一化的注意力得分：
    - $\omega_{21} = x^{(1)} q^{(2)\top}$
    - $\omega_{22} = x^{(2)} q^{(2)\top}$
    - $\omega_{23} = x^{(3)} q^{(2)\top}$
    - ...
    - $\omega_{2T} = x^{(T)} q^{(2)\top}$
- 上述式子中，$\omega$ 是希腊字母 omega，用于表示未归一化的注意力得分
    - $\omega_{21}$ 中的下标 "21" 表示使用输入序列元素 2 作为查询去匹配输入序列元素 1


- 假设我们有如下输入句子，且已按照第 2 章的方法嵌入为 3 维向量（为了便于展示且不发生换行，这里使用了非常小的嵌入维度）：


In [2]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your（你的）    (x^1)
   [0.55, 0.87, 0.66], # journey（旅程） (x^2)
   [0.57, 0.85, 0.64], # starts（始于）  (x^3)
   [0.22, 0.58, 0.33], # with（足下）    (x^4)
   [0.77, 0.25, 0.10], # one（一）       (x^5)
   [0.05, 0.80, 0.55]] # step（步）      (x^6)
)

- （在本书中，我们遵循机器学习和深度学习的通用惯例，将训练样本表示为行，特征值表示为列；在上文展示的张量中，每一行代表一个单词，每一列代表一个嵌入维度）

- 本节的主要目标是演示如何以第二个输入序列元素 $x^{(2)}$ 作为查询来计算上下文向量 $z^{(2)}$

- 下图描绘了该过程的第一步，即通过点积操作计算 $x^{(2)}$ 与所有其他输入元素之间的注意力得分 $\omega$


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/08.webp" width="800px">

- 我们以输入序列元素 2（即 $x^{(2)}$）为例来计算上下文向量 $z^{(2)}$；稍后在本节中，我们将把该计算推广到所有上下文向量。
- 第一步是通过计算查询 $x^{(2)}$ 与所有其他输入 token 之间的点积，来求出未归一化的注意力得分：


In [3]:
query = inputs[1]  # 第 2 个输入 token 作为查询 (query)

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query) # 点积（因为它们是一维向量，此处无需转置）

print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


- 补充说明：点积本质上是两个向量对应元素相乘并将乘积相加的简写形式：


In [4]:
res = 0.

for idx, element in enumerate(inputs[0]):
    res += inputs[0][idx] * query[idx]

print(res)
print(torch.dot(inputs[0], query))

tensor(0.9544)
tensor(0.9544)


- **步骤 2：** 对未归一化的注意力得分（即 omega，$\omega$）进行归一化，使其总和为 1
- 下面是将未归一化注意力得分归一化为总和为 1 的一种简单方法（这是一种标准惯例，便于直观解释，且对模型训练稳定性至关重要）：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/09.webp" width="800px">

In [5]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()

print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


- 然而在实践中，通常推荐使用 softmax 函数进行归一化，因为 softmax 更擅长处理极值，并且在模型训练过程中具有更优良的梯度性质。
- 下面是用于缩放的 softmax 函数的朴素实现，它同样将向量元素归一化使得总和为 1：


In [6]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)

print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- 上述朴素实现对于过大或过小的输入值可能会由于数值上溢（overflow）和下溢（underflow）而出现数值不稳定性问题
- 因此在实际工程中，建议使用 PyTorch 官方实现的 softmax，它已经针对性能与数值稳定性进行了高度优化：


In [7]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


- **步骤 3：** 将嵌入的输入 token $x^{(i)}$ 与对应的注意力权重相乘，并对所得向量求和，从而计算出上下文向量 $z^{(2)}$：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/10.webp" width="800px">

In [8]:
query = inputs[1] # 第 2 个输入 token 作为查询 (query)

context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i]*x_i

print(context_vec_2)

tensor([0.4419, 0.6515, 0.5683])


&nbsp;
### 3.3.2 计算所有输入 token 的注意力权重


#### 推广到输入序列的所有 token：

- 上面我们计算了输入 2 的注意力权重和上下文向量（如下方图中高亮行所示）
- 接下来，我们将把这种计算推广到所有注意力权重与上下文向量的计算


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/11.webp" width="800px">

- （请注意，为了减少视觉干扰，该图中的数字截断保留两位小数；每行中的数值之和应为 1.0 或 100%；类似地，其他图中的数字也做了截断）


- 在自注意力机制中，整个流程始于注意力得分的计算，随后对其进行归一化以获得总和为 1 的注意力权重
- 然后利用这些注意力权重，通过对输入的加权求和来生成上下文向量


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/12.webp" width="800px">

- 将先前的**步骤 1**应用到所有成对元素上，计算出未归一化的注意力得分矩阵：


In [9]:
attn_scores = torch.empty(6, 6)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attn_scores[i, j] = torch.dot(x_i, x_j)

print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 我们可以通过矩阵乘法更高效地实现上述操作：


In [10]:
attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


- 与此前的**步骤 2**类似，我们对每一行进行归一化，使得每一行中的数值总和为 1：


In [11]:
attn_weights = torch.softmax(attn_scores, dim=-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


- 快速验证每一行中的数值之和确实为 1：


In [12]:
row_2_sum = sum([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
print("Row 2 sum:", row_2_sum)

print("All row sums:", attn_weights.sum(dim=-1))

Row 2 sum: 1.0
All row sums: tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


- 应用先前的**步骤 3**来计算所有的上下文向量：


In [13]:
all_context_vecs = attn_weights @ inputs
print(all_context_vecs)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


- 作为合理性检查，之前计算的上下文向量 $z^{(2)} = [0.4419, 0.6515, 0.5683]$ 可以在上面的第 2 行中找到：


In [14]:
print("Previous 2nd context vector:", context_vec_2)

Previous 2nd context vector: tensor([0.4419, 0.6515, 0.5683])


&nbsp;
## 3.4 实现带可训练权重的自注意力机制


- 概念结构图：说明本节开发的自注意力机制如何融入本书和本章的整体脉络与架构中


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/13.webp" width="800px">

&nbsp;
### 3.4.1 逐步计算注意力权重


- 在本节中，我们将实现原始 Transformer 架构、GPT 模型以及大多数其他流行 LLM 中所使用的自注意力机制
- 这种自注意力机制也被称为“缩放点积注意力 (scaled dot-product attention)”
- 整体思路与之前类似：
  - 我们希望将上下文向量计算为针对特定输入元素的输入向量加权和
  - 为此，我们需要注意力权重
- 正如你将看到的，与前面介绍的基础注意力机制相比，只有细微的差别：
  - 最显著的区别是引入了在模型训练期间更新的权重矩阵
  - 这些可训练的权重矩阵至关重要，它们使模型（具体来说是模型内部的注意力模块）能够学会生成“良好”的上下文向量


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/14.webp" width="800px">

- 逐步实现自注意力机制，我们首先介绍三个可训练权重矩阵 $W_q$、$W_k$ 和 $W_v$
- 这三个矩阵用于通过矩阵乘法将嵌入的输入 token $x^{(i)}$ 投影为查询（query）、键（key）和值（value）向量：

  - 查询向量：$q^{(i)} = x^{(i)}\,W_q $
  - 键向量：$k^{(i)} = x^{(i)}\,W_k $
  - 值向量：$v^{(i)} = x^{(i)}\,W_v $


- 输入 $x$ 与查询向量 $q$ 的嵌入维度可以相同也可以不同，具体取决于模型设计与特定实现
- 在 GPT 模型中，输入与输出维度通常相同，但为了便于教学演示、更清楚地跟踪计算过程，我们在这里选择不同的输入与输出维度：


In [15]:
x_2 = inputs[1] # 第 2 个输入元素
d_in = inputs.shape[1] # 输入嵌入维度大小，d=3
d_out = 2 # 输出嵌入维度大小，d=2

- 下面我们初始化这三个权重矩阵；请注意，为了在演示中减少输出中的干扰信息，我们设置了 `requires_grad=False`；但如果是用于模型训练，则需要设置 `requires_grad=True` 以便在训练过程中更新这些矩阵：


In [16]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key   = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

- 接下来我们计算查询、键和值向量：


In [17]:
query_2 = x_2 @ W_query # 下标 _2 表示它是针对第 2 个输入元素的
key_2 = x_2 @ W_key 
value_2 = x_2 @ W_value

print(query_2)

tensor([0.4306, 1.4551])


- 如下所示，我们成功地将 6 个输入 token 从 3 维嵌入空间投影到了 2 维嵌入空间：


In [18]:
keys = inputs @ W_key 
values = inputs @ W_value

print("keys.shape:", keys.shape)
print("values.shape:", values.shape)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


- 在下一步（**步骤 2**）中，我们通过计算查询向量与每个键向量之间的点积来求出未归一化的注意力得分：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/15.webp" width="800px">

In [19]:
keys_2 = keys[1] # Python 索引从 0 开始
attn_score_22 = query_2.dot(keys_2)
print(attn_score_22)

tensor(1.8524)



- 因为我们有 6 个输入，所以对于给定的查询向量，我们得到 6 个注意力得分：


In [20]:
attn_scores_2 = query_2 @ keys.T # 给定查询对应的所有注意力得分
print(attn_scores_2)

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/16.webp" width="800px">

- 接下来，在**步骤 3**中，我们使用前面用过的 softmax 函数来计算注意力权重（总和归一化为 1 的注意力得分）
- 与之前的区别在于，现在我们通过将注意力得分除以嵌入维度的平方根 $\sqrt{d_k}$（即 `d_k**0.5`）来进行缩放：


In [21]:
d_k = keys.shape[1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
print(attn_weights_2)

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/17.webp" width="800px">

- 在**步骤 4**中，我们现在为输入查询向量 2 计算上下文向量：


In [22]:
context_vec_2 = attn_weights_2 @ values
print(context_vec_2)

tensor([0.3061, 0.8210])


&nbsp;
### 3.4.2 实现紧凑的 SelfAttention 类


总结对照  
对象	定义	权重张量形状 (shape)	示例数值 (d_in=3, d_out=2)
v1.W_query	nn.Parameter	(d_in, d_out)	torch.Size([3, 2])
v2.W_query.weight	nn.Linear	(d_out, d_in)	torch.Size([2, 3])

NOTE

虽然底层的权重张量维度是转置的，但经过前向传播投影后，得到的 queries 张量形状是一致的，均为 (batch_size, num_tokens, d_out)（或单样本下的 (num_tokens, d_out)）。

- 整合上述所有步骤，我们可以将自注意力机制实现如下：


#### 3.4.2.1 SelfAttention_v1

In [23]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):

    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key   = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        
        attn_scores = queries @ keys.T # 未归一化的注意力得分 omega
        attn_weights = torch.softmax(
            attn_scores / keys.shape[-1]**0.5, dim=-1
        )

        context_vec = attn_weights @ values
        return context_vec

torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)

print ("--- SelfAttention_v1 ---")
print("queries.shape:", sa_v1.W_query.shape)
print("SelfAttention_v1 output:", sa_v1(inputs))

--- SelfAttention_v1 ---
queries.shape: torch.Size([3, 2])


SelfAttention_v1 output: tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/18.webp" width="800px">

#### 3.4.2.2 ※SelfAttention_v2

- 我们可以利用 PyTorch 的 Linear 层来精简上述实现；如果不启用偏置单元（bias），Linear 层在数学上完全等价于矩阵乘法
- 相比手动使用 `nn.Parameter(torch.rand(...)`，使用 `nn.Linear` 的另一个巨大优势在于它具备更合理的权重初始化策略，能带来更稳定的模型训练效果：


In [24]:
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        assert(id(x) == id(inputs))  # 是那个全局的inputs

        # 这些 self._ 前缀的属性只是为了调试使用，本身不需要修改为成员变量
        self._queries = self.W_query(x)
        self._keys = self.W_key(x)
        self._values = self.W_value(x)

        # W_key 是一个 nn.Linear 对象，其计算存在2个等价形势
        keys2 = torch.nn.functional.linear(x, self.W_key.weight, self.W_key.bias)
        print("keys2 == self._keys:", torch.allclose(keys2, self._keys))
        # keys3 = x @ self.W_key.weight.T + self.W_key.bias if self.W_key.bias is not None else 0
  
        
        self._attn_scores = self._queries @ self._keys.T
        self._attn_weights = torch.softmax(self._attn_scores / self._keys.shape[-1]**0.5, dim=-1)

        context_vec = self._attn_weights @ self._values
        return context_vec

torch.manual_seed(123)
sa_v2 = SelfAttention_v2(d_in, d_out)

print("------ SelfAttention_v2 ------")
print("inputs:", inputs)
print("queries.shape:", sa_v2.W_query.weight.shape)
print("context_vec:", sa_v2(inputs))

------ SelfAttention_v2 ------
inputs: tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
queries.shape: torch.Size([2, 3])
keys2 == self._keys: True
context_vec: tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)


##### SelfAttention_v2 的矩阵关系

`self.W_query` 是一个 `nn.Linear` 层；`self.W_query.weight` 是可训练的权重矩阵，而 `self._queries` 是输入经过该层计算得到的查询矩阵。

下图以当前示例的 `n=6`、`d_in=3`、`d_out=2`、`qkv_bias=False` 和 `torch.manual_seed(123)` 展示完整流程：输入 `X` → `Q/K/V` → `Q @ K.T` → 缩放及逐行 `softmax` → `A @ V`。图中数值由当前 `SelfAttention_v2` 实际计算。

![SelfAttention_v2 矩阵关系说明图](images/selfattention-v2-matrices.png)

[高清 PNG](images/selfattention-v2-matrices.png) · [SVG 矢量图](images/selfattention-v2-matrices.svg)

##### 将 attn_scores 展开为输入与权重的乘法

下面的推导对应当前默认的 `qkv_bias=False`，即三个 Linear 层均不使用偏置。令 `X` 为输入，`WQ`、`WK` 分别为查询层与键层内部的权重矩阵：

```python
X = x                         # (n, d_in)
WQ = self.W_query.weight       # (d_out, d_in)
WK = self.W_key.weight         # (d_out, d_in)

Q = X @ WQ.T                  # (n, d_out)
K = X @ WK.T                  # (n, d_out)
attn_scores = Q @ K.T          # (n, n)
```

注意 `nn.Linear` 的 `.weight` 按 `(d_out, d_in)` 存储，因此输入右乘权重的转置。将 `Q` 与 `K` 代入注意力分数公式：

$$
\begin{aligned}
\text{attn\_scores}
&= QK^\top \\
&= (XW_Q^\top)(XW_K^\top)^\top \\
&= (XW_Q^\top)(W_KX^\top) \\
&= \boxed{XW_Q^\top W_KX^\top}.
\end{aligned}
$$

其中用到了**转置整个乘积时，交换乘法顺序**的规则：

$$
(AB)^\top = B^\top A^\top,
\qquad
(XW_K^\top)^\top = (W_K^\top)^\top X^\top = W_KX^\top.
$$

因此，`X @ WK.T` 得到 `K`，而 `WK @ X.T` 得到 `K.T`；两者互为转置，当前示例中的形状分别为 `(6, 2)` 和 `(2, 6)`。

展开后的代码为：

```python
attn_scores = X @ WQ.T @ WK @ X.T
```

在当前 `n=6`、`d_in=3`、`d_out=2` 的示例中，各矩阵的维度依次为：

```text
   X        WQ.T       WK        X.T
(6 × 3) @ (3 × 2) @ (2 × 3) @ (3 × 6) → (6 × 6)
```

也可以先合并两套权重，令 `M = WQ.T @ WK`，其形状为 `(d_in, d_in)`，当前为 `(3, 3)`，再计算 `attn_scores = X @ M @ X.T`。每个分数满足 $S_{ij} = x_i M x_j^\top$，表示第 $i$ 个 token 与第 $j$ 个 token 通过这个由权重决定的矩阵进行匹配。


##### 将上下文输出 C 展开为基本矩阵

沿用上文的 `qkv_bias=False`，再令 `WV = self.W_value.weight`，形状同样为 `(d_out, d_in)`。值矩阵与注意力权重分别为：

$$
V = XW_V^\top,
\qquad
A = \operatorname{softmax}_{\mathrm{row}}\!\left(
\frac{XW_Q^\top W_KX^\top}{\sqrt{d_{\mathrm{out}}}}
\right).
$$

这里 $\operatorname{softmax}_{\mathrm{row}}$ 表示沿每一行的列方向归一化，对应 `torch.softmax(..., dim=-1)`；每行的权重之和为 1。缩放因子使用 Q/K 的特征维度 `d_out`，当前示例为 $\sqrt{2}$。

将 `A` 和 `V` 代入最终输出 `C = A @ V`，得到仅由输入与三套权重组成的公式：

$$
\begin{aligned}
C
&= AV \\
&= \operatorname{softmax}_{\mathrm{row}}\!\left(
\frac{QK^\top}{\sqrt{d_{\mathrm{out}}}}
\right)(XW_V^\top) \\
&= \boxed{
\operatorname{softmax}_{\mathrm{row}}\!\left(
\frac{XW_Q^\top W_KX^\top}{\sqrt{d_{\mathrm{out}}}}
\right)XW_V^\top
}.
\end{aligned}
$$

对应代码为：

```python
X = x
WQ = self.W_query.weight
WK = self.W_key.weight
WV = self.W_value.weight
d_out = WQ.shape[0]

C = torch.softmax(
    (X @ WQ.T @ WK @ X.T) / (d_out ** 0.5),
    dim=-1,
) @ (X @ WV.T)
```

`softmax` 将 `(n, n)` 的分数矩阵转换为同形状的权重矩阵，之后与 `(n, d_out)` 的值矩阵相乘，输出 `C` 的形状为 `(n, d_out)`。当前示例中的最后一步为：

```text
   A          V          C
(6 × 6) @ (6 × 2) → (6 × 2)
```

得到 `A` 后，还可以按矩阵乘法结合律写成 $C = (AX)W_V^\top$，即先按注意力权重对输入进行加权聚合，再做值投影：

```text
   A          X         WV.T
(6 × 6) @ (6 × 3) @ (3 × 2) → (6 × 2)
```

其中 `A` 本身依赖当前输入 `X` 以及 `WQ`、`WK`，完整计算中保留了逐行 `softmax`。


##### X 到上下文输出 C 的矩阵转换流程

下图展示同一个输入 `X` 如何通过 `WQ`、`WK`、`WV` 生成 `Q`、`K`、`V`，以及 `Q @ K.T` 经缩放和逐行 `softmax` 得到 `A`，最终通过 `C = A @ V` 得到上下文输出。

实线表示数据变换与传递，虚线表示权重参与计算；各节点标注当前示例的矩阵维度，图底部给出了只使用 `X` 与三套权重的展开公式。

![SelfAttention_v2：X 与各矩阵的转换流程和关系](images/selfattention-v2-flow.png)

[高清 PNG](images/selfattention-v2-flow.png) · [SVG 矢量图](images/selfattention-v2-flow.svg)

- 请注意，`SelfAttention_v1` 和 `SelfAttention_v2` 会给出不同的输出，这是因为它们的权重矩阵采用了不同的初始权重


调试

In [25]:
print("------ SelfAttention_v2 ------")
print("inputs:", inputs)
print("W_query:", sa_v2.W_query.weight.data)
print("W_key:", sa_v2.W_key.weight.data)
print("W_value:", sa_v2.W_value.weight.data)

print("_queries:", sa_v2._queries)
print("_keys:", sa_v2._keys)
print("_values:", sa_v2._values)

print("attn_scores:", sa_v2._attn_scores)
print("attn_weights:", sa_v2._attn_weights)

print("context_vec1:", sa_v2(inputs))


------ SelfAttention_v2 ------
inputs: tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])
W_query: tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]])
W_key: tensor([[-0.4196, -0.4590, -0.3648],
        [ 0.2615, -0.2133,  0.2161]])
W_value: tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]])
_queries: tensor([[-0.3536,  0.3965],
        [-0.3021, -0.0289],
        [-0.3015, -0.0232],
        [-0.1353, -0.0978],
        [-0.2052,  0.0870],
        [-0.1542, -0.1499]], grad_fn=<MmBackward0>)
_keys: tensor([[-0.5740,  0.2727],
        [-0.8709,  0.1008],
        [-0.8628,  0.1060],
        [-0.4789,  0.0051],
        [-0.4744,  0.1696],
        [-0.5888, -0.0388]], grad_fn=<MmBackward0>)
_values: tensor([[-0.4519,  0.2216],
        [-0.7142, -0.1961],
        [-0.7127, -0.1971],
        [-0.

In [26]:
print("------")


------


&nbsp;
## 3.5 使用因果注意力遮蔽未来词元


- 在因果注意力（causal attention）中，对角线上方的注意力权重被掩码遮蔽，从而确保对于任何给定的输入，LLM 在利用注意力权重计算上下文向量时都无法访问未来的 token：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/19.webp" width="800px">

&nbsp;
### 3.5.1 应用因果注意力掩码


- 在本节中，我们将把之前的自注意力机制转换为因果自注意力机制
- 因果自注意力确保模型对序列中某个特定位置的预测仅依赖于前面位置已知的输出，而不依赖于未来位置
- 换言之，这保证了下一个词的预测只能取决于它前面的词
- 为实现这一点，对于每个给定的 token，我们需要遮蔽未来的 token（即输入文本中位于当前 token 之后的 token）：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/20.webp" width="800px">

- 为了演示并实现因果自注意力，我们沿用上一节中的注意力得分与权重：


In [27]:
# 为方便起见，复用上一节中
# SelfAttention_v2 对象的查询与键权重矩阵
queries = sa_v2.W_query(inputs)
keys = sa_v2.W_key(inputs) 
attn_scores = queries @ keys.T

attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
print("Attention weights:\n", attn_weights)

Attention weights:
 tensor([[0.1717, 0.1762, 0.1761, 0.1555, 0.1627, 0.1579],
        [0.1636, 0.1749, 0.1746, 0.1612, 0.1605, 0.1652],
        [0.1637, 0.1749, 0.1746, 0.1611, 0.1606, 0.1651],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.1632, 0.1674],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.1639],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


- 遮蔽未来注意力权重最简单的方法是使用 PyTorch 的 tril 函数创建一个掩码：主对角线及下方的元素设为 1，主对角线上方的元素设为 0：


In [28]:
context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
print(mask_simple)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


- 然后，我们可以将注意力权重与此掩码相乘，将对角线上方的注意力得分置零：


In [29]:
masked_simple = attn_weights*mask_simple
print(masked_simple)

tensor([[0.1717, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1749, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1749, 0.1746, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1704, 0.1702, 0.1652, 0.0000, 0.0000],
        [0.1667, 0.1722, 0.1721, 0.1618, 0.1633, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<MulBackward0>)


- 然而，如果像上面那样在 softmax 之后应用掩码，会破坏 softmax 所生成的概率分布
- Softmax 能确保所有输出值之和为 1
- 在 softmax 之后进行掩码遮蔽需要重新将输出归一化为总和为 1，这不仅增加了计算的繁琐度，还可能引发意外副作用


- 为了确保每行的总和仍为 1，我们可以如下重新归一化注意力权重：


In [30]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple / row_sums
print("Normalized masked simple:\n", masked_simple_norm)

Normalized masked simple:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<DivBackward0>)


- 虽然从技术上讲我们现在已经完成了因果注意力机制的编写，但我们来简要了解一种更高效的等价实现方法
- 与其将对角线上方的注意力权重置零并重新归一化，我们可以在未归一化的注意力得分传入 softmax 函数之前，将对角线上方的得分直接掩码替换为负无穷大（负无穷在经过 softmax 后其权重值自然变为 0）：


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/21.webp" width="600px">

In [31]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)


- 如下所示，现在每一行中的注意力权重再次正确地归一化为总和为 1：


In [32]:
attn_weights = torch.softmax(masked / keys.shape[-1]**0.5, dim=-1)
print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)


&nbsp;
### 3.5.2 使用 Dropout 进一步遮蔽注意力权重


- 此外，我们还应用了 Dropout 来减轻训练过程中的过拟合现象
- Dropout 可以应用于多个位置：
  - 例如在计算注意力权重之后；
  - 或者在注意力权重与值向量相乘之后
- 在这里，我们选择在计算注意力权重之后应用 Dropout 掩码，因为这种做法更为常见

- 此外，在当前具体示例中，我们使用了 50% 的 dropout 率，这意味着随机遮蔽掉一半的注意力权重。（后续在训练 GPT 模型时，我们将使用更低的 dropout 率，例如 0.1 或 0.2）


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/22.webp" width="800px">

- 如果我们应用 0.5（50%）的 dropout 率，未被丢弃的值将按比例放大，缩放因子为 1/0.5 = 2
- 缩放比例依据公式 1 / (1 - `dropout_rate`) 计算


In [33]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(0.5) # 丢弃率为 50%
example = torch.ones(6, 6) # 创建全 1 矩阵

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [34]:
torch.manual_seed(123)
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)


- 请注意，根据操作系统的不同，生成的 Dropout 输出可能会有所不同；有关此不一致性的更多信息，请参阅 [PyTorch issue 跟踪器](https://github.com/pytorch/pytorch/issues/121595)


&nbsp;
### 3.5.3 ※实现紧凑的因果自注意力类


- 现在，我们已经准备好实现一个包含因果掩码和 Dropout 掩码的完整可用自注意力类
- 还需要补充的一点是增加处理多样本批次（batch）的代码，以便我们的 `CausalAttention` 类能够支持第 2 章中实现的数据加载器所输出的批次数据
- 为简单起见，我们通过复制输入文本示例来模拟此类批次输入：


In [35]:
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape) # 包含 2 个输入样本，每个样本含 6 个 token，且每个 token 的嵌入维度为 3

torch.Size([2, 6, 3])



In [36]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout) # 新增
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1)) # 新增



    def forward(self, x):
        print("Begin [CausalAttention.forward]")
        b, num_tokens, d_in = x.shape # 新增批次维度 b

        # 若输入的 `num_tokens` 超过 `context_length`，
        # 将在下方创建掩码时引发错误。
        # 在实践中这并不是问题，因为 LLM（第 4-7 章）会确保输入在进入该 forward 方法前  
        # 长度不会超出 `context_length`。 
        
        self._queries = self.W_query(x)
        self._keys = self.W_key(x)
        self._values = self.W_value(x)

        self._attn_scores = self._queries @ self._keys.transpose(1, 2) # 调整了转置维度
        
        # 新增，带下划线的方法为就地 (in-place) 操作
        # `:num_tokens` 用于适配批次中的 token 数量小于支持的最大上下文大小 (context_size) 的情况
        self._attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)  
        
        self._attn_weights = torch.softmax(self._attn_scores / self._keys.shape[-1]**0.5, dim=-1)
        self._attn_weights = self.dropout(self._attn_weights) # 新增

        context_vec = self._attn_weights @ self._values

        print("context_length =", context_length)

        print("d_in =", d_in)
        print("d_out =", d_out)
        assert id(x) == id(batch), "Input must be the same object as the global batch"

        # 可调参数
        print("x shape:", x.shape)
        print(x)
        print("self.W_query =\n", self.W_query.weight.data, sep="")
        print("self.W_key =\n", self.W_key.weight.data, sep="")
        print("self.W_value =\n", self.W_value.weight.data, sep="")
        print("dropout =", self.dropout)
        print("self.mask.data =\n", self.mask.data, sep="")
        print("")

        # forward 方法中计算的中间结果
        print("\nself._queries.shape:", self._queries.shape)
        print("self._queries:\n", self._queries, sep="")
        print("\nself._keys.shape:", self._keys.shape)
        print("self._keys:\n", self._keys, sep="")
        print("\nself._values.shape:", self._values.shape)
        print("self._values:\n", self._values, sep="")
        print("\nself._attn_scores.shape:", self._attn_scores.shape)
        print("self._attn_scores:\n", self._attn_scores, sep="")
        print("self._attn_weights.shape:", self._attn_weights.shape)
        print("\nself._attn_weights:\n", self._attn_weights, sep="")

        print("context_vecs.shape:", context_vec.shape)
        print("\ncontext_vec:\n", context_vec, sep="")
        print("End [CausalAttention.forward]\n")

        return context_vec

print("Begin [CausalAttention]")
torch.manual_seed(123)
context_length = batch.shape[1]
ca = CausalAttention(d_in, d_out, context_length, 0.0)

context_vecs = ca(batch)

print("End [CausalAttention]")

Begin [CausalAttention]
Begin [CausalAttention.forward]
context_length = 6
d_in = 3
d_out = 2
x shape: torch.Size([2, 6, 3])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
self.W_query =
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]])
self.W_key =
tensor([[-0.4196, -0.4590, -0.3648],
        [ 0.2615, -0.2133,  0.2161]])
self.W_value =
tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]])
dropout = Dropout(p=0.0, inplace=False)
self.mask.data =
tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0.,

调试信息

In [37]:
print("--- CausalAttention ---")
print("batch.shape:", batch.shape)
print("batch:", batch)


print("W_query:", ca.W_query.weight.data)
print("W_key:", ca.W_key.weight.data)
print("W_value:", ca.W_value.weight.data)

#_queries
print("_queries:", ca._queries)
print("_keys:", ca._keys)
print("_values:", ca._values)

print("attn_scores:", ca._attn_scores)
print("attn_weights:", ca._attn_weights)

print("context_vec1:", ca(batch))

--- CausalAttention ---
batch.shape: torch.Size([2, 6, 3])
batch: tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
W_query: tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]])
W_key: tensor([[-0.4196, -0.4590, -0.3648],
        [ 0.2615, -0.2133,  0.2161]])
W_value: tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]])
_queries: tensor([[[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.3015, -0.0232],
         [-0.1353, -0.0978],
         [-0.2052,  0.0870],
         [-0.1542, -0.1499]],

        [[-0.3536,  0.3965],
         [-0.3021, -0.0289],
         [-0.301

- 请注意，Dropout 仅在模型训练时启用，在推理（inference）阶段不启用


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/23.webp" width="800px">

&nbsp;
## 3.6 将单头注意力扩展为多头注意力


&nbsp;
### 3.6.1 堆叠多个单头注意力层


- 下面是前面实现的自注意力机制的概括总结（为简洁起见，未展示因果掩码和 Dropout 掩码）

- 这也被称为单头注意力（single-head attention）：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/24.webp" width="600px">

- 我们只需堆叠多个单头注意力模块即可获得多头注意力模块：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/25.webp" width="600px">

- 多头注意力的核心思想是使用不同的、学习到的线性投影并行运行多次注意力机制。这使得模型能够在不同位置共同关注来自不同表示子空间的信息。


In [38]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        # self.heads = nn.ModuleList(
        #     [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias) for _ in range(num_heads)]
        # )
        # 列表推导式（List Comprehension）
        # [ <表达式> for <循环变量> in <可迭代对象> ]
        print("num_heads =", num_heads)
        heads_list = []
        for _ in range(num_heads):
            head = CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
            heads_list.append(head)
        self.heads = nn.ModuleList(heads_list)

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


print("Begin [MultiHeadAttentionWrapper]")

torch.manual_seed(123)

context_length = batch.shape[1] # 这是 token 的数量
d_in, d_out = 3, 2
mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print("====\ncontext_vecs.shape:", context_vecs.shape)
print("context_vecs:\n", context_vecs)

# 当前实际4位精度的输出参考，用于对比参考
mha_context_vecs = torch.tensor([
    [
        [-0.4519,  0.2216,  0.4772,  0.1063],
        [-0.5874,  0.0058,  0.5891,  0.3257],
        [-0.6300, -0.0632,  0.6202,  0.3860],
        [-0.5675, -0.0843,  0.5478,  0.3589],
        [-0.5526, -0.0981,  0.5321,  0.3428],
        [-0.5299, -0.1081,  0.5077,  0.3493]
    ],
    [
        [-0.4519,  0.2216,  0.4772,  0.1063],
        [-0.5874,  0.0058,  0.5891,  0.3257],
        [-0.6300, -0.0632,  0.6202,  0.3860],
        [-0.5675, -0.0843,  0.5478,  0.3589],
        [-0.5526, -0.0981,  0.5321,  0.3428],
        [-0.5299, -0.1081,  0.5077,  0.3493]
    ]
])

#PyTorch 内部有一个默认配置： torch.set_printoptions(precision=4)
print("IS EQUAL:", torch.allclose(mha_context_vecs, context_vecs, atol=1e-4))
#print(context_vecs[0, 5, 3])        # tensor(0.3493, grad_fn=<SelectBackward0>)
#print(context_vecs[0, 5, 3].item()) # 0.3493485152721405

print("End [MultiHeadAttentionWrapper]")

Begin [MultiHeadAttentionWrapper]
num_heads = 2
Begin [CausalAttention.forward]
context_length = 6
d_in = 3
d_out = 2
x shape: torch.Size([2, 6, 3])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
self.W_query =
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]])
self.W_key =
tensor([[-0.4196, -0.4590, -0.3648],
        [ 0.2615, -0.2133,  0.2161]])
self.W_value =
tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]])
dropout = Dropout(p=0.0, inplace=False)
self.mask.data =
tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],


- 在上述实现中，输出嵌入维度为 4，因为我们设置了 `d_out=2` 作为键、查询、值向量以及上下文向量的嵌入维度。既然我们有 2 个注意力头，因此总的输出嵌入维度为 2*2=4


&nbsp;
### 3.6.2 ※通过权重拆分实现多头注意力


- 虽然上面对先前单头注意力 `CausalAttention` 进行包装的实现既直观又功能完备，但我们可以编写一个独立的 `MultiHeadAttention` 类来实现相同的效果

- 在这个独立的 `MultiHeadAttention` 类中，我们不再拼接单独的注意力头
- 相反，我们创建单个 W_query、W_key 和 W_value 权重矩阵，然后将它们拆分为每个注意力头各自的独立部分：


In [46]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 缩减投影维度以匹配预期的输出维度

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # 用于合并各注意力头输出的线性层
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # 类似于 `CausalAttention`，若输入的 `num_tokens` 超过 `context_length`， 
        # 将在下方创建掩码时引发错误。 
        # 在实践中这并不是问题，因为 LLM（第 4-7 章）会确保输入在进入该 forward 方法前  
        # 长度不会超出 `context_length`。
        print("Begin [MultiHeadAttention.forward]")     
        assert x is batch, "x and batch must be the same object"

        _queries1 = self.W_query(x)
        _keys1 = self.W_key(x) # 形状：(b, num_tokens, d_out)
        _values1 = self.W_value(x)

        # 通过增加 `num_heads` 维度隐式拆分矩阵
        # 展开最后一个维度：(b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        _queries2 = _queries1.view(b, num_tokens, self.num_heads, self.head_dim)
        _keys2 = _keys1.view(b, num_tokens, self.num_heads, self.head_dim) 
        _values2 = _values1.view(b, num_tokens, self.num_heads, self.head_dim)

        # 转置变换：(b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        queries = _queries2.transpose(1, 2)
        keys = _keys2.transpose(1, 2)
        values = _values2.transpose(1, 2)

        # 计算带因果掩码的缩放点积注意力（即自注意力）
        attn_scores = queries @ keys.transpose(2, 3)  # 每个注意力头分别计算点积

        # 将原始掩码截断至实际 token 数量并转换为布尔型
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 使用掩码填充注意力得分
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 形状：(b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # 合并所有注意力头，其中 self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # 可选的投影层

        print("context_length =", context_length)
        print("d_in =", d_in)
        print("d_out =", d_out)
        print("head_dim =", self.head_dim)
        print("num_heads =", self.num_heads)

        assert id(x) == id(batch), "Input must be the same object as the global batch"

        # 可调参数
        print("x shape:", x.shape)
        print(x)
        print("self.W_query =\n", self.W_query.weight.data, sep="")
        print("self.W_key =\n", self.W_key.weight.data, sep="")
        print("self.W_value =\n", self.W_value.weight.data, sep="")
        print("dropout =", self.dropout)
        print("self.mask.data =\n", self.mask.data, sep="")
        print("")

        # forward 方法中计算的中间结果
        print("\n_queries1.shape:", _queries1.shape)
        print("_queries1:\n", _queries1, sep="")

        print("\n_queries2.shape:", _queries2.shape)
        print("_queries2:\n", _queries2, sep="")

        print("queries.shape:", queries.shape)
        print("queries:\n", queries, sep="")
        print("keys.shape:", keys.shape)
        print("keys:\n", keys, sep="")
        print("\nvalues.shape:", values.shape)
        print("values:\n", values, sep="")
        print("\nattn_scores.shape:", attn_scores.shape)
        print("attn_scores:\n", attn_scores, sep="")
        print("attn_weights.shape:", attn_weights.shape)
        print("\nattn_weights:\n", attn_weights, sep="")

        print("\ncontext_vecs.shape:", context_vec.shape)
        print("context_vec:\n", context_vec)
        print("End [MultiHeadAttention.forward]\n")

        return context_vec

print("Begin [MultiHeadAttention]")
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print("End [MultiHeadAttention]")

Begin [MultiHeadAttention]
Begin [MultiHeadAttention.forward]
context_length = 6
d_in = 3
d_out = 2
head_dim = 1
num_heads = 2
x shape: torch.Size([2, 6, 3])
tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])
self.W_query =
tensor([[-0.2354,  0.0191, -0.2867],
        [ 0.2177, -0.4919,  0.4232]])
self.W_key =
tensor([[-0.4196, -0.4590, -0.3648],
        [ 0.2615, -0.2133,  0.2161]])
self.W_value =
tensor([[-0.4900, -0.3503, -0.2120],
        [-0.1135, -0.4404,  0.3780]])
dropout = Dropout(p=0.0, inplace=False)
self.mask.data =
tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 

- 请注意，上述实现本质上是 `MultiHeadAttentionWrapper` 的重写版本，执行效率更高
- 由于随机权重初始化不同，两者的输出略有差异，但它们都是功能完备的实现，均可用于我们在后续章节中实现的 GPT 类中


---

**关于输出维度的说明**

- 在上述 `MultiHeadAttention` 中，我使用了 `d_out=2`，以便与前面 `MultiHeadAttentionWrapper` 类的设置保持一致
- `MultiHeadAttentionWrapper` 由于进行了拼接，其输出的头维度为 `d_out * num_heads`（即 `2*2 = 4`）
- 然而，`MultiHeadAttention` 类（为了更加易用）允许我们直接通过 `d_out` 控制输出维度；这意味着如果我们设置 `d_out = 2`，无论注意力头有多少个，输出头维度都将是 2
- 事后看来，正如读者在 [PR #859](https://github.com/rasbt/LLMs-from-scratch/pull/859) 中指出的那样，将 `MultiHeadAttention` 设置为 `d_out = 4` 可能更直观，这样它能产生与 `d_out = 2` 的 `MultiHeadAttentionWrapper` 相同的输出维度。

---


- 另外请注意，我们在上面的 `MultiHeadAttention` 类中添加了一个线性投影层（`self.out_proj`）。这只是一个不改变维度的线性变换。在 LLM 实现中使用这种投影层是标准惯例，但它并非严格必需（近期的研究表明，移除该层并不会影响建模性能；详见本章末尾的扩展阅读部分）


<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch03_compressed/26.webp" width="800px">

- 请注意，如果你对上述逻辑的紧凑且高效的官方实现感兴趣，也可以参考 PyTorch 中的 [`torch.nn.MultiheadAttention`](https://pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html) 类


- 上述实现初看可能略显复杂，我们来看看执行 `attn_scores = queries @ keys.transpose(2, 3)` 时底层具体发生了什么：


In [40]:
# 维度示例：(b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

print(a @ a.transpose(2, 3))

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


- 在这种情况下，PyTorch 中的矩阵乘法实现能够自动处理 4 维输入张量，使得矩阵乘法在最后两个维度（num_tokens, head_dim）之间进行，并在各个独立的注意力头之间重复执行

- 例如，以下代码展示了分别为每个头单独计算矩阵乘法的一种更紧凑的书写方式：


In [41]:
first_head = a[0, 0, :, :]
first_res = first_head @ first_head.T
print("First head:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\nSecond head:\n", second_res)

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


&nbsp;
## 本章小结与收获


- 请参阅 [./multihead-attention.ipynb](./multihead-attention.ipynb) 代码 Notebook，它是数据加载器（第 2 章）加上本章实现的多头注意力类的精简合并版本，我们将在后续章节中用它来训练 GPT 模型
- 本章习题解答请参阅 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)
